In [5]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
import pickle
import pandas as pd 
from sklearn.model_selection import train_test_split

In [6]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [8]:
##encode the categotical result
label_encoder_gender=LabelEncoder()

data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

In [9]:
##Similar for the geography column
onehot_encoder_geo=OneHotEncoder()
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [11]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [12]:
data.describe()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,650.528800,0.545700,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700,0.501400,0.250900,0.247700
std,96.653299,0.497932,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769,0.500023,0.433553,0.431698
min,350.000000,0.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000,0.000000,0.000000,0.000000
25%,584.000000,0.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000,0.000000,0.000000,0.000000
50%,652.000000,1.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000,1.000000,0.000000,0.000000
75%,718.000000,1.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000,1.000000,1.000000,0.000000
max,850.000000,1.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000,1.000000,1.000000,1.000000


In [13]:
x=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [14]:
X_train,X_test,Y_train,Y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [15]:
scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [16]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)
    

## ANN Regression Problem Statement

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [18]:
## Build the model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1) ###output layer for regression (By the default the activation function is linear)
     
])

#### Compile the model
model.compile(optimizer='adam',loss='mean_squared_error',metrics=['mae'])    ###mae stands for Mean Absolute Error.

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
## set up TensorBoard
log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)
                                                      

In [20]:
#### Set up Early stopping 
early_stopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [21]:
### Train the model
history=model.fit(
    X_train,Y_train,
    validation_data=(X_test,Y_test),
    epochs=100,
    callbacks=[early_stopping,tensorboard_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 6ms/step - loss: 13383834624.0000 - mae: 100377.5781 - val_loss: 13008147456.0000 - val_mae: 98523.6250
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 13238080512.0000 - mae: 99652.4844 - val_loss: 12718636032.0000 - val_mae: 97059.8594
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 12721883136.0000 - mae: 97100.7344 - val_loss: 11975593984.0000 - val_mae: 93296.6875
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 11696830464.0000 - mae: 92035.4062 - val_loss: 10723742720.0000 - val_mae: 86970.1875
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 10192413696.0000 - mae: 84549.7578 - val_loss: 9082167296.0000 - val_mae: 78678.5391
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 8408548864.0000 - mae: 75642.2344 - val_loss: 7311174144.0000 - val_mae: 69821.3203
Epoch 7/100
250/250 [=============

In [22]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [26]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 16932), started 0:02:29 ago. (Use '!kill 16932' to kill it.)

In [24]:
"""conda activate venv
pip install --upgrade setuptools"""

'conda activate venv\npip install --upgrade setuptools'

In [27]:
### Evaluate the model
test_loss,test_mae=model.evaluate(X_test,Y_test)
print(f'Test Loss: {test_loss}, Test MAE: {test_mae}')

63/63 [==============================] - 4s 21ms/step - loss: 3365000448.0000 - mae: 50167.9883
Test Loss: 3365000448.0, Test MAE: 50167.98828125


In [28]:
model.save('regression_model.h5')

c:\Users\Lenovo\Desktop\PROJECT ANN\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
